In [3]:
import csv
import json
import os
import shutil
from collections import Counter

# --- 1. Configuration & Data Generation Rules ---
STUDENT_ID_STRING = "TUPM-26-4336"
SEED = 4336  # Numeric portion of your ID
SEED_DIGIT = SEED % 10

# Rule: First 1000 + ID
RECORDS_TO_PROCESS = 1000 + SEED

# Rule: (SEED_DIGIT % 4) + 2
STATUS_CLASS_NUM = (SEED_DIGIT % 4) + 2
TARGET_STATUS_PREFIX = str(STATUS_CLASS_NUM)  # This will be "4" for 4xx

# File paths
INPUT_FILE = 'weblog.csv'
FILTERED_CSV = 'filtered_records.csv'
SUMMARY_JSON = 'summary.json'
BACKUP_JSON = 'summary_backup.json'

# --- 2. Initialization ---
status_code_counts = Counter()
url_counts = Counter()
filtered_rows = []
records_processed = 0

print(f"Student ID: {SEED}")
print(f"Target Status Class: {TARGET_STATUS_PREFIX}xx")
print(f"Records to Process: {RECORDS_TO_PROCESS}")
print("-" * 30)

# --- 3. File Processing (Line by Line) ---
try:
    with open(INPUT_FILE, mode='r', encoding='utf-8', newline='') as infile:
        reader = csv.DictReader(infile)
        
        # FIXED: Changed 'Status' to 'Staus' to match the dataset's actual typo
        required_cols = {'IP', 'Time', 'URL', 'Staus'}
        if not required_cols.issubset(reader.fieldnames):
            print(f"Error: CSV missing required columns. Found: {reader.fieldnames}")
            exit(1)

        for row in reader:
            if records_processed >= RECORDS_TO_PROCESS:
                break
            
            records_processed += 1
            
            # Extract fields (using the typo 'Staus' for the dictionary key)
            ip = row.get('IP', '').strip()
            timestamp = row.get('Time', '').strip()
            url = row.get('URL', '').strip()
            status = row.get('Staus', '').strip()
            
            # Check if status belongs to target class (e.g., starts with "4")
            if status.startswith(TARGET_STATUS_PREFIX):
                # Store for CSV export
                filtered_rows.append({
                    'IP': ip,
                    'Time': timestamp,
                    'URL': url,
                    'Staus': status  # Keeping the typo here so the output CSV matches original
                })
                
                # Update counters for summary
                status_code_counts[status] += 1
                url_counts[url] += 1

except FileNotFoundError:
    print(f"Error: The file '{INPUT_FILE}' was not found. Please ensure it is in the current directory.")
    exit(1)
except Exception as e:
    print(f"An error occurred while reading the file: {e}")
    exit(1)

# --- 4. Analysis & Output Generation ---

# Get top 5 URLs
top_5_urls = url_counts.most_common(5)

# Prepare Summary Dictionary
summary_data = {
    "student_id": SEED,
    "records_processed": records_processed,
    "target_status_class": f"{TARGET_STATUS_PREFIX}xx",
    "records_in_target_class": len(filtered_rows),
    "count_per_status_code": dict(status_code_counts),
    "top_5_urls": [{"url": url, "count": count} for url, count in top_5_urls]
}

# Requirement 5: Export Filtered CSV
try:
    with open(FILTERED_CSV, mode='w', encoding='utf-8', newline='') as outfile:
        fieldnames = ['IP', 'Time', 'URL', 'Staus']
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(filtered_rows)
    print(f"Success: Filtered records written to '{FILTERED_CSV}'")
except Exception as e:
    print(f"Error writing CSV: {e}")

# Requirement 5: Write Summary JSON
try:
    with open(SUMMARY_JSON, mode='w', encoding='utf-8') as jsonfile:
        json.dump(summary_data, jsonfile, indent=4)
    print(f"Success: Summary written to '{SUMMARY_JSON}'")
except Exception as e:
    print(f"Error writing JSON: {e}")

# --- 5. Backup and Verification (Requirement 6) ---
try:
    # Create backup
    shutil.copy2(SUMMARY_JSON, BACKUP_JSON)
    
    # Verify backup exists
    if os.path.exists(BACKUP_JSON):
        print(f"Success: Backup created and verified at '{BACKUP_JSON}'")
    else:
        print(f"Error: Backup file '{BACKUP_JSON}' was not found after creation.")
        
except FileNotFoundError:
    print("Error: Cannot create backup because the source JSON file is missing.")
except Exception as e:
    print(f"Error during backup process: {e}")

# --- 6. Final Output (Assessment Data) ---
print("\n" + "=" * 30)
print("ASSESSMENT DATA")
print("=" * 30)
print(f"Records processed: {summary_data['records_processed']}")
print(f"Target status class: {summary_data['target_status_class']}")
print(f"Records in target class: {summary_data['records_in_target_class']}")
print(f"Count per status code: {summary_data['count_per_status_code']}")
print(f"Five most requested URLs:")
for item in summary_data['top_5_urls']:
    print(f"  - {item['url']} ({item['count']} requests)")
print(f"Filtered CSV file: {FILTERED_CSV}")
print(f"JSON summary file: {SUMMARY_JSON}")
print(f"Backup verification: {'Passed' if os.path.exists(BACKUP_JSON) else 'Failed'}")

Student ID: 4336
Target Status Class: 4xx
Records to Process: 5336
------------------------------
Success: Filtered records written to 'filtered_records.csv'
Success: Summary written to 'summary.json'
Success: Backup created and verified at 'summary_backup.json'

ASSESSMENT DATA
Records processed: 5336
Target status class: 4xx
Records in target class: 69
Count per status code: {'404': 69}
Five most requested URLs:
  - GET /robots.txt HTTP/1.1 (57 requests)
  - GET /favicon.ico HTTP/1.1 (8 requests)
  - GET /djs/vendor/bootstrap-datetimepicker.js HTTP/1.1 (3 requests)
  - GET /css/bootstrap.min.css.map HTTP/1.1 (1 requests)
Filtered CSV file: filtered_records.csv
JSON summary file: summary.json
Backup verification: Passed


In [4]:
import csv
import json
import os
import shutil

# --- 1. Configuration & Data Generation Rules ---
SEED = 4336  # Your numeric Student ID
SEED_DIGIT = SEED % 10

# Rule 1: Android if SEED_DIGIT is even; iOS if odd
# 4336 % 10 = 6 (Even) -> Target OS is Android
TARGET_OS = "Android" if (SEED_DIGIT % 2 == 0) else "iOS"

# Rule 2: User Behavior Class target
# (SEED_DIGIT % 5) + 1 -> (6 % 5) + 1 = 1 + 1 = 2
TARGET_BEHAVIOR_CLASS = (SEED_DIGIT % 5) + 1

# File paths
INPUT_FILE = 'user_behavior_dataset.csv'
EXPORTED_CSV = 'exported_users.csv'
SUMMARY_JSON = 'usage_summary.json'
BACKUP_CSV = 'exported_users_backup.csv'

# --- 2. Initialization ---
filtered_records = []
total_records_in_dataset = 0
records_matching_os = 0
target_class_records = 0

battery_sum = 0.0
data_usage_sum = 0.0

print(f"Student ID: {SEED}")
print(f"Target Operating System: {TARGET_OS}")
print(f"Target User Behavior Class: {TARGET_BEHAVIOR_CLASS}")
print("-" * 40)

# --- 3. File Processing ---
try:
    with open(INPUT_FILE, mode='r', encoding='utf-8', newline='') as infile:
        # Use DictReader but handle potential leading/trailing spaces in headers
        reader = csv.DictReader(infile)
        
        # Clean the headers to avoid KeyError if the CSV has spaces (e.g., " User ID ")
        cleaned_fieldnames = [field.strip() for field in reader.fieldnames]
        
        # We need to map the cleaned headers to the actual row keys
        for row in reader:
            total_records_in_dataset += 1
            
            # Create a clean dictionary for this row
            clean_row = {k.strip(): v.strip() for k, v in row.items() if k is not None}
            
            # Extract the OS (Handle possible "Android" vs "android")
            current_os = clean_row.get('Operating System', '').strip().capitalize()
            
            # Requirement 1: Identify records matching the assigned OS
            if current_os == TARGET_OS:
                records_matching_os += 1
                
                try:
                    # Extract User Behavior Class (Handle if it's saved as "2" or "2.0" or "Class 2")
                    behavior_str = clean_row.get('User Behavior Class', '0')
                    # Convert to float first to handle "2.0", then to int
                    current_behavior_class = int(float(behavior_str))
                except ValueError:
                    current_behavior_class = 0 # Default fallback if data is messy
                
                # Requirement 2: From these, select users belonging to target class
                if current_behavior_class == TARGET_BEHAVIOR_CLASS:
                    target_class_records += 1
                    
                    # Extract required fields for export
                    user_id = clean_row.get('User ID', '')
                    device_model = clean_row.get('Device Model', '')
                    battery_drain_str = clean_row.get('Battery Drain (mAh/day)', '0')
                    data_usage_str = clean_row.get('Data Usage (MB/day)', '0')
                    
                    # Calculate sums for averages
                    try:
                        battery_drain = float(battery_drain_str)
                        data_usage = float(data_usage_str)
                        
                        battery_sum += battery_drain
                        data_usage_sum += data_usage
                    except ValueError:
                        battery_drain = 0
                        data_usage = 0
                    
                    # Store for CSV export
                    filtered_records.append({
                        'User ID': user_id,
                        'Device Model': device_model,
                        'Operating System': TARGET_OS,
                        'Battery Drain (mAh/day)': battery_drain_str,
                        'Data Usage (MB/day)': data_usage_str
                    })

except FileNotFoundError:
    print(f"Error: The file '{INPUT_FILE}' was not found. Please download it from Kaggle and place it in this folder.")
    exit(1)

# --- 4. Analysis (Requirement 3: Calculate Averages) ---
if target_class_records > 0:
    avg_battery = battery_sum / target_class_records
    avg_data = data_usage_sum / target_class_records
else:
    avg_battery = 0
    avg_data = 0

# --- 5. Output Generation ---
# Requirement 4: Export to CSV
try:
    with open(EXPORTED_CSV, mode='w', encoding='utf-8', newline='') as outfile:
        fieldnames = ['User ID', 'Device Model', 'Operating System', 'Battery Drain (mAh/day)', 'Data Usage (MB/day)']
        writer = csv.DictWriter(outfile, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(filtered_records)
except Exception as e:
    print(f"Error writing CSV: {e}")

# Requirement 5: Write JSON summary
summary_data = {
    "student_id": SEED,
    "operating_system": TARGET_OS,
    "target_behavior_class": TARGET_BEHAVIOR_CLASS,
    "total_records_in_dataset": total_records_in_dataset,
    "records_matching_os": records_matching_os,
    "records_in_target_class": target_class_records,
    "average_battery_drain": round(avg_battery, 2),
    "average_data_usage": round(avg_data, 2)
}

with open(SUMMARY_JSON, mode='w', encoding='utf-8') as jsonfile:
    json.dump(summary_data, jsonfile, indent=4)

# --- 6. Backup and Verification (Requirement 6) ---
try:
    shutil.copy2(EXPORTED_CSV, BACKUP_CSV)
    backup_exists = os.path.exists(BACKUP_CSV)
    backup_size = os.path.getsize(BACKUP_CSV) if backup_exists else 0
except Exception as e:
    backup_exists = False
    backup_size = 0

# --- 7. Final Output (Assessment Data) ---
print("\n" + "=" * 40)
print("ASSESSMENT DATA")
print("=" * 40)
print(f"Operating system and behavior class: {TARGET_OS}, Class {TARGET_BEHAVIOR_CLASS}")
print(f"Total records in dataset: {total_records_in_dataset}")
print(f"Records matching operating system: {records_matching_os}")
print(f"Records in target behavior class: {target_class_records}")
print(f"Average battery drain (mAh/day): {round(avg_battery, 2)}")
print(f"Average data usage (MB/day): {round(avg_data, 2)}")

# Requirement 5: Verify exported count matches selected record count
if len(filtered_records) == target_class_records:
    print(f"Count verification: Passed (Exported: {len(filtered_records)}, Expected: {target_class_records})")
else:
    print(f"Count verification: FAILED (Exported: {len(filtered_records)}, Expected: {target_class_records})")

print(f"Exported CSV file: {EXPORTED_CSV}")
print(f"JSON summary file: {SUMMARY_JSON}")
print(f"Backup file and size: {'Exists' if backup_exists else 'Missing'} ({backup_size} bytes)")

Student ID: 4336
Target Operating System: Android
Target User Behavior Class: 2
----------------------------------------

ASSESSMENT DATA
Operating system and behavior class: Android, Class 2
Total records in dataset: 700
Records matching operating system: 554
Records in target behavior class: 114
Average battery drain (mAh/day): 887.11
Average data usage (MB/day): 451.44
Count verification: Passed (Exported: 114, Expected: 114)
Exported CSV file: exported_users.csv
JSON summary file: usage_summary.json
Backup file and size: Exists (4117 bytes)


In [5]:
import csv
import json
import os
import shutil

# --- 1. Configuration & Data Generation Rules ---
SEED = 4336
SEED_DIGIT = SEED % 10

TARGET_YEAR = 2015 + SEED_DIGIT

# File paths
INPUT_FILE = 'Global_Cybersecurity_Threats_2015-2024.csv'
FILTERED_CSV = 'filtered_incidents.csv'
SUMMARY_JSON = 'incident_summary.json'
BACKUP_JSON = 'incident_summary_backup.json'
REPORT_TXT = 'incident_report.txt'

# --- 2. Initialization ---
filtered_records = []
total_records = 0
target_year_incidents = 0
attack_counts = {}
total_loss = 0.0
valid_loss_count = 0

print(f"Target Year: {TARGET_YEAR}")
print("-" * 40)

# --- 3. File Processing ---
try:
    with open(INPUT_FILE, mode='r', encoding='utf-8', newline='') as infile:
        reader = csv.DictReader(infile)
        
        # Clean headers to avoid issues with leading/trailing spaces in the CSV
        for row in reader:
            total_records += 1
            clean_row = {k.strip(): v.strip() for k, v in row.items() if k is not None}
            
            year_str = clean_row.get('Year', '')
            
            # Requirement 1: Select incidents occurring in target year
            if year_str == str(TARGET_YEAR):
                target_year_incidents += 1
                
                attack_type = clean_row.get('Attack Type', 'Unknown')
                loss_str = clean_row.get('Financial Loss (in Million $)', '0')
                
                # Requirement 2: Count incidents per attack type
                if attack_type:
                    attack_counts[attack_type] = attack_counts.get(attack_type, 0) + 1
                
                # Handle invalid financial values
                try:
                    loss_val = float(loss_str)
                except ValueError:
                    loss_val = 0.0 # Treat invalid as 0 for total calculation
                
                # Requirement 3: Calculate total loss
                total_loss += loss_val
                valid_loss_count += 1
                
                # Store record for sorting and export
                record_to_add = {
                    'Country': clean_row.get('Country', ''),
                    'Year': year_str,
                    'Attack Type': attack_type,
                    'Financial Loss (in Million $)': loss_str,
                    'Number of Affected Users': clean_row.get('Number of Affected Users', ''),
                    'Attack Source': clean_row.get('Attack Source', ''),
                    'Security Vulnerability Type': clean_row.get('Security Vulnerability Type', ''),
                    'Defense Mechanism Used': clean_row.get('Defense Mechanism Used', ''),
                    'loss_float': loss_val # Keep float for sorting internally
                }
                filtered_records.append(record_to_add)

except FileNotFoundError:
    print(f"Error: The file '{INPUT_FILE}' was not found.")
    exit(1)

# --- 4. Analysis (Requirements 3 & 4) ---
# Calculate Average
if valid_loss_count > 0:
    avg_loss = total_loss / valid_loss_count
else:
    avg_loss = 0.0

# Requirement 4: Identify five costliest incidents
# We sort descending by 'loss_float'. Python's sort is stable, so ties retain original order.
sorted_records = sorted(filtered_records, key=lambda x: x['loss_float'], reverse=True)
top_5_costliest = sorted_records[:5]

# Clean up the internal sorting key before saving to JSON/CSV
for r in filtered_records:
    r.pop('loss_float', None)
for r in top_5_costliest:
    r.pop('loss_float', None)

# --- 5. Output Generation ---
# Requirement 5: Export filtered CSV
try:
    if filtered_records:
        with open(FILTERED_CSV, mode='w', encoding='utf-8', newline='') as outfile:
            # Exclude the temporary 'loss_float' key
            fieldnames = [k for k in filtered_records[0].keys() if k != 'loss_float']
            writer = csv.DictWriter(outfile, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(filtered_records)
    else:
        # Handle empty results appropriately
        with open(FILTERED_CSV, mode='w', encoding='utf-8') as outfile:
            outfile.write("No incidents found for the target year.\n")
except Exception as e:
    print(f"Error writing CSV: {e}")

# Requirement 5: Export JSON Summary
summary_data = {
    "target_year": TARGET_YEAR,
    "total_records_in_dataset": total_records,
    "incidents_in_target_year": target_year_incidents,
    "count_per_attack_type": attack_counts,
    "total_financial_loss": round(total_loss, 2),
    "average_financial_loss": round(avg_loss, 2),
    "five_costliest_incidents": top_5_costliest
}

try:
    with open(SUMMARY_JSON, mode='w', encoding='utf-8') as jsonfile:
        json.dump(summary_data, jsonfile, indent=4)
except Exception as e:
    print(f"Error writing JSON: {e}")

# Requirement 5: Backup and verify JSON
backup_success = False
try:
    shutil.copy2(SUMMARY_JSON, BACKUP_JSON)
    backup_success = os.path.exists(BACKUP_JSON)
except Exception as e:
    print(f"Error during backup: {e}")

# Requirement 6: Generate Text Report
try:
    with open(REPORT_TXT, mode='w', encoding='utf-8') as txtfile:
        txtfile.write("CYBERSECURITY INCIDENT REPORT\n")
        txtfile.write("=" * 30 + "\n")
        txtfile.write(f"Target Year: {TARGET_YEAR}\n")
        txtfile.write(f"Total Incidents Found: {target_year_incidents}\n\n")
        
        txtfile.write("INCIDENT COUNTS PER ATTACK TYPE:\n")
        for atk, count in attack_counts.items():
            txtfile.write(f" - {atk}: {count}\n")
        txtfile.write("\n")
        
        txtfile.write("FIVE COSTLIEST INCIDENTS:\n")
        for i, inc in enumerate(top_5_costliest, 1):
            txtfile.write(f"{i}. {inc['Attack Type']} in {inc['Country']} - ${inc['Financial Loss (in Million $)']}M\n")
        txtfile.write("\n")
        
        txtfile.write("OUTPUT FILES:\n")
        txtfile.write(f" - Filtered CSV: {FILTERED_CSV}\n")
        txtfile.write(f" - Summary JSON: {SUMMARY_JSON}\n")
        txtfile.write(f" - Backup JSON: {BACKUP_JSON}\n")
except Exception as e:
    print(f"Error writing report: {e}")


# --- 6. Final Output (Assessment Data) ---
print("\n" + "=" * 40)
print("ASSESSMENT DATA")
print("=" * 40)
print(f"Target year: {TARGET_YEAR}")
print(f"Total records in dataset: {total_records}")
print(f"Incidents in target year: {target_year_incidents}")
print(f"Count per attack type: {attack_counts}")
print(f"Total financial loss (USD millions): {round(total_loss, 2)}")
print(f"Average financial loss (USD millions): {round(avg_loss, 2)}")
print(f"Five costliest incidents:")
for i, inc in enumerate(top_5_costliest, 1):
    print(f"  {i}. {inc['Attack Type']} ({inc['Country']}) - ${inc['Financial Loss (in Million $)']}M")
print(f"Filtered CSV file: {FILTERED_CSV}")
print(f"JSON summary file: {SUMMARY_JSON}")
print(f"Backup verification: {'Passed' if backup_success else 'Failed'}")
print(f"Report file: {REPORT_TXT}")

Target Year: 2021
----------------------------------------

ASSESSMENT DATA
Target year: 2021
Total records in dataset: 3000
Incidents in target year: 299
Count per attack type: {'Phishing': 41, 'SQL Injection': 59, 'Malware': 49, 'DDoS': 47, 'Ransomware': 55, 'Man-in-the-Middle': 48}
Total financial loss (USD millions): 15873.41
Average financial loss (USD millions): 53.09
Five costliest incidents:
  1. DDoS (USA) - $99.81M
  2. Ransomware (France) - $99.77M
  3. SQL Injection (France) - $98.76M
  4. SQL Injection (China) - $98.57M
  5. SQL Injection (India) - $98.09M
Filtered CSV file: filtered_incidents.csv
JSON summary file: incident_summary.json
Backup verification: Passed
Report file: incident_report.txt
